In [ ]:
import psycopg2
from sentence_transformers import SentenceTransformer

# ----------------------------
# 1. Load embedding model
# ----------------------------
# all-MiniLM-L6-v2 produces dense vector embeddings of size 384
# These embeddings capture semantic meaning of text (not just keywords)
# They will be stored in PostgreSQL using the pgvector extension
model = SentenceTransformer('all-MiniLM-L6-v2')

# ----------------------------
# 2. Movie data
# ----------------------------
# Sample dataset: each movie has a title and a description
# The description will be converted into an embedding vector
movies_data = [
    {"title": "Space Adventure", "description": "A thrilling space journey to distant galaxies."},
    {"title": "Romantic Comedy", "description": "A light-hearted romance with humorous twists."},
    {"title": "Sci-Fi Thriller", "description": "A scientist explores new planets and faces unknown dangers."},
    {"title": "Action Hero", "description": "An action-packed adventure with thrilling stunts."},
    {"title": "Mystery Unfolded", "description": "A suspenseful story of intrigue and surprises."},
    {"title": "Fantasy Realm", "description": "An epic tale set in a magical world."},
    {"title": "Horror Nights", "description": "A terrifying tale of fear and survival in the dark."},
    {"title": "Comedy Club", "description": "A hilarious collection of stand-up performances."},
    {"title": "Historical Drama", "description": "A moving story set during a significant historical event."},
    {"title": "Animated Adventure", "description": "A whimsical journey through a colorful animated world."},
    {"title": "Detective Noir", "description": "A gritty detective story filled with twists and turns."},
    {"title": "Superhero Saga", "description": "An epic battle between heroes and villains."},
    {"title": "Family Reunion", "description": "A comedic drama about the ups and downs of family gatherings."},
    {"title": "Epic Quest", "description": "A journey through mystical lands in search of a legendary artifact."},
    {"title": "Thrilling Escape", "description": "A heart-pounding story of survival and escape from danger."},
    {"title": "Romantic Drama", "description": "A deep exploration of love and relationships."},
    {"title": "Underwater Odyssey", "description": "An exploration of the ocean's depths and its mysteries."},
    {"title": "Time Travel Adventure", "description": "A thrilling journey through different eras and events."},
    {"title": "Zombie Apocalypse", "description": "A fight for survival in a world overrun by the undead."},
    {"title": "Musical Extravaganza", "description": "A colorful and lively musical filled with song and dance."},
    {"title": "Spy Thriller", "description": "A gripping story of espionage and betrayal."},
]

# ----------------------------
# 3. Database configuration (PostgreSQL)
# ----------------------------
# Connection details for PostgreSQL database
# Replace placeholders with your actual credentials
# sslmode="require" ensures a secure encrypted connection
DB_CONFIG = {
    "host": "<YOUR_HOST>",
    "port": "5432",
    "database": "<YOUR_DATABASE>",
    "user": "<YOUR_USERNAME>",
    "password": "<YOUR_PASSWORD>",
    "sslmode": "require"
}

conn = None
cur = None

try:
    # ----------------------------
    # 4. Connect to PostgreSQL
    # ----------------------------
    # Establish connection to PostgreSQL database using psycopg2
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    print("Connection successful to PostgreSQL!")

    # ----------------------------
    # 5. Database Setup (pgvector)
    # ----------------------------
    # Enable pgvector extension (required for storing embeddings)
    # This allows PostgreSQL to store and operate on vector data types
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    
    # Drop table if it already exists (for clean re-run)
    cur.execute("DROP TABLE IF EXISTS movies;")
    
    # Create table to store:
    # - title (text)
    # - description (text)
    # - embedding (vector of size 384)
    # The embedding column uses pgvector type
    cur.execute("""
        CREATE TABLE movies (
            id SERIAL PRIMARY KEY,
            title TEXT,
            description TEXT,
            embedding vector(384)
        );
    """)
    conn.commit()
    print("Table created successfully in PostgreSQL.")

    # ----------------------------
    # 6. Generate embeddings and insert into PostgreSQL
    # ----------------------------
    print("Generating embeddings and inserting into PostgreSQL...")

    # Loop through each movie
    for movie in movies_data:
        # Convert movie description into a 384-dimensional embedding vector
        embedding = model.encode(movie["description"]).tolist()

        # Insert into PostgreSQL table
        # embedding is stored in pgvector format
        cur.execute("""
            INSERT INTO movies (title, description, embedding)
            VALUES (%s, %s, %s)
        """, (
            movie["title"],
            movie["description"],
            embedding
        ))

    # Commit transaction to persist data in PostgreSQL
    conn.commit()
    print("Movies and embeddings stored successfully in PostgreSQL.")

except Exception as e:
    # Catch and print any errors (connection issues, SQL errors, etc.)
    print(f"Error: {e}")

finally:
    # ----------------------------
    # 7. Cleanup
    # ----------------------------
    # Close cursor and connection to PostgreSQL
    if cur:
        cur.close()
    if conn:
        conn.close()
    print("\nPostgreSQL connection closed.")
